# 05. Подготовка агрегатов для Yandex DataLens

##  Цель
Сформировать набор "плоских" CSV-файлов, которые позволят построить в DataLens **интерактивный дашборд** с кросс-фильтрацией.

## Логика подготовки
- Все агрегаты считаются **в PostgreSQL** — быстро, без выгрузки миллионов строк в pandas.
- pandas используется только как транспорт: `read_sql` → проверка → `to_csv`.
- Кодировка `utf-8-sig` — чтобы Excel и DataLens корректно читали кириллицу и специальные символы.

## Структура датасетов

| # | Файл | Назначение | Тип |
|---|------|------------|-----|
| 1 | `dl_kpi.csv` | KPI-карточки | Агрегат (1 строка) |
| 2 | `dl_by_year.csv` | Динамика по годам | Агрегат (год) |
| 3 | `dl_by_genre.csv` | Статистика по жанрам | Агрегат (жанр) |
| 4 | `dl_by_genre_combo.csv` | Комбо-жанры | Агрегат (пара жанров) |
| 5 | `dl_by_region.csv` | Статистика по регионам | Агрегат (регион) |
| 6 | `dl_region_genre.csv` | Связка регион + жанр | Агрегат (регион × жанр) |
| 7 | `dl_top_titles.csv` | Топ-100 фильмов | Детализация (фильм) |
| 8 | `dl_top_people.csv` | Топ людей | Агрегат (человек) |
| 9 | `dl_movie_cast_region.csv` | **Фильм + человек + регион** | **Связка (для кликов)** |
| 10 | `dl_movie_genre.csv` | **Фильм + жанр** | **Связка (для кликов)** |


In [ ]:
import os
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv(Path("../.env"))

DB_URL = (
    f"postgresql+psycopg://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
engine = create_engine(DB_URL, connect_args={"options": "-csearch_path=imdb"})

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def export(df: pd.DataFrame, filename: str) -> None:
    path = PROCESSED_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"{filename}: rows={len(df)}, cols={len(df.columns)}")


def check(df: pd.DataFrame, name: str) -> None:
    if df.empty:
        raise ValueError(f"{name} пустой")
    if df.isnull().sum().sum():
        print(f"{name}: есть NaN")

## KPI-карточки (`dl_kpi.csv`)

In [5]:
query_kpi = """
SELECT
    (SELECT COUNT(*) FROM title_basics) AS total_titles,
    (SELECT COUNT(*) FROM title_basics WHERE titletype = 'movie') AS total_movies,
    (SELECT COUNT(*) FROM title_basics WHERE titletype = 'tvSeries') AS total_series,
    (SELECT COUNT(*) FROM name_basics) AS total_people,
    (SELECT ROUND(AVG(averagerating)::numeric, 2) FROM title_ratings r JOIN title_basics b ON b.tconst = r.tconst WHERE b.titletype = 'movie') AS avg_movie_rating,
    (SELECT SUM(numvotes) FROM title_ratings r JOIN title_basics b ON b.tconst = r.tconst WHERE b.titletype = 'movie') AS total_movie_votes;
"""
df_kpi = pd.read_sql(query_kpi, engine)
check(df_kpi, "df_kpi")
export(df_kpi, "dl_kpi.csv")

dl_kpi.csv: rows=1, cols=6


## Динамика по годам (`dl_by_year.csv`)

Для линейного графика: сколько фильмов выходило каждый год + средний рейтинг + суммарные голоса.

**Фильтр:** `startyear BETWEEN 1900 AND 2025` — чтобы отсечь аномалии (единичные фильмы 1800-х и будущие релизы).

In [6]:
query_by_year = """
SELECT
    b.startyear AS year,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating,
    SUM(r.numvotes) AS total_votes
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
WHERE b.titletype = 'movie' AND b.startyear BETWEEN 1900 AND 2025
GROUP BY b.startyear
ORDER BY b.startyear;
"""
df_by_year = pd.read_sql(query_by_year, engine)
check(df_by_year, "df_by_year")
export(df_by_year, "dl_by_year.csv")

dl_by_year.csv: rows=126, cols=5


In [16]:
df_by_year = pd.read_sql(query_by_year, engine)
check(df_by_year, "df_by_year")

print(f"Лет: {len(df_by_year)}")
print(f"Диапазон: {df_by_year['year'].min()}–{df_by_year['year'].max()}")
print(f"Пик: {df_by_year.loc[df_by_year['movies_count'].idxmax(), 'year']} "
      f"({df_by_year['movies_count'].max():,} фильмов)")
print(f"Средний рейтинг за всё время: {df_by_year['avg_rating'].mean():.2f}")
df_by_year.head()

Лет: 126
Диапазон: 1900–2025
Пик: 2023 (11,954 фильмов)
Средний рейтинг за всё время: 5.85


,year,movies_count,avg_rating,median_rating,total_votes
0,1900,6,3.72,3.6,179
1,1901,8,3.58,3.6,189
2,1902,3,2.53,2.6,50
3,1903,4,4.23,3.9,921
4,1904,3,4.47,5.3,624


## Статистика по жанрам (`dl_by_genre.csv`)

Для бар-чарта: сколько фильмов в каждом жанре + средний и медианный рейтинг.

**Фильтр:** `numvotes >= 6000` — чтобы отсечь фильмы с 1-2 голосами, которые искажают среднее.

In [7]:
query_by_genre = """
SELECT
    g.genre_name AS genre,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating,
    ROUND(STDDEV(r.averagerating)::numeric, 2) AS std_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_genres tg ON tg.tconst = b.tconst
JOIN genres g ON g.genre_id = tg.genre_id
WHERE b.titletype = 'movie' AND r.numvotes >= 6000
GROUP BY g.genre_name
ORDER BY movies_count DESC;
"""
df_by_genre = pd.read_sql(query_by_genre, engine)
check(df_by_genre, "df_by_genre")
export(df_by_genre, "dl_by_genre.csv")  

df_by_genre: есть NaN
dl_by_genre.csv: rows=24, cols=5


## Комбо-жанры (`dl_by_genre_combo.csv`)

Self-join `title_genres` с условием `g1.genre_id < g2.genre_id` — чтобы не задваивать пары и не строить (Drama, Drama).

**Фильтр:** `HAVING COUNT >= 100` — иначе в дашборде будут видны редкие комбо с шумным рейтингом.

In [8]:
query_combo = """
SELECT
    g1.genre_name AS genre_1,
    g2.genre_name AS genre_2,
    g1.genre_name || ' + ' || g2.genre_name AS combo,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating,
    ROUND(STDDEV(r.averagerating)::numeric, 2) AS std_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_genres tg1 ON tg1.tconst = b.tconst
JOIN title_genres tg2 ON tg2.tconst = b.tconst
JOIN genres g1 ON g1.genre_id = tg1.genre_id
JOIN genres g2 ON g2.genre_id = tg2.genre_id
WHERE b.titletype = 'movie' AND g1.genre_id < g2.genre_id
GROUP BY g1.genre_name, g2.genre_name
HAVING COUNT(DISTINCT b.tconst) >= 100
ORDER BY movies_count DESC;
"""
df_combo = pd.read_sql(query_combo, engine)
check(df_combo, "df_combo")
export(df_combo, "dl_by_genre_combo.csv")

dl_by_genre_combo.csv: rows=164, cols=7


## Статистика по регионам (`dl_by_region.csv`)

Из `title_akas` берём регион для каждого тайтла. Один тайтл может иметь несколько `region` — считаем все.

**Фильтр:** `HAVING COUNT >= 500` — чтобы убрать страны с 1-2 фильмами.

In [9]:
query_by_region = """
SELECT
    ak.region,
    COUNT(DISTINCT b.tconst) AS titles_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_akas ak ON ak.titleid = b.tconst
WHERE b.titletype = 'movie' AND ak.region IS NOT NULL
GROUP BY ak.region
HAVING COUNT(DISTINCT b.tconst) >= 500
ORDER BY titles_count DESC;
"""
df_by_region = pd.read_sql(query_by_region, engine)
check(df_by_region, "df_by_region")
export(df_by_region, "dl_by_region.csv")

dl_by_region.csv: rows=85, cols=3


## Связка регион + жанр (`dl_region_genre.csv`)

Для тепловой карты: какие жанры популярны в каких регионах.

**Фильтр:** `HAVING COUNT >= 100` — чтобы не было шума.

In [10]:
query_region_genre = """
SELECT
    ak.region,
    g.genre_name AS genre,
    COUNT(DISTINCT b.tconst) AS movies_count,
    ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
JOIN title_akas ak ON ak.titleid = b.tconst
JOIN title_genres tg ON tg.tconst = b.tconst
JOIN genres g ON g.genre_id = tg.genre_id
WHERE b.titletype = 'movie' AND ak.region IS NOT NULL
GROUP BY ak.region, g.genre_name
HAVING COUNT(DISTINCT b.tconst) >= 100
ORDER BY ak.region, movies_count DESC;
"""
df_region_genre = pd.read_sql(query_region_genre, engine)
check(df_region_genre, "df_region_genre")
export(df_region_genre, "dl_region_genre.csv")

dl_region_genre.csv: rows=1539, cols=5


## Топ-100 фильмов (`dl_top_titles.csv`)

Топ по рейтингу с фильтром по голосам — чтобы в таблицу не попали «фильмы с 10.0 и 50 голосами».

**Порог:** `numvotes >= 100000`.

In [11]:
query_top_titles = """
SELECT
    b.tconst, b.primarytitle AS title, b.startyear AS year,
    b.genres, b.runtimeminutes AS runtime,
    r.averagerating AS rating, r.numvotes AS votes
FROM title_basics b
JOIN title_ratings r ON r.tconst = b.tconst
WHERE b.titletype = 'movie' AND r.numvotes >= 100000
ORDER BY r.averagerating DESC, r.numvotes DESC
LIMIT 100;
"""
df_top_titles = pd.read_sql(query_top_titles, engine)
check(df_top_titles, "df_top_titles")
export(df_top_titles, "dl_top_titles.csv")

dl_top_titles.csv: rows=100, cols=7


## Топ людей (`dl_top_people.csv`)

Два отдельных запроса → конкатенация в один датасет с колонкой `role`.

**Пороги:**
- Актёры: `numvotes >= 70000`, `movies_count >= 10`
- Режиссёры: `numvotes >= 70000`, `movies_count >= 5`

**Метрика:** `median_rating` — не среднее. Один провальный фильм не должен обрушивать медиану.

In [12]:
query_top_people = """
WITH actors AS (
    SELECT n.primaryname AS person, 'actor' AS role, COUNT(DISTINCT b.tconst) AS movies_count,
           ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    JOIN title_principals p ON p.tconst = b.tconst
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 70000 AND p.category IN ('actor', 'actress')
    GROUP BY n.primaryname
    HAVING COUNT(DISTINCT b.tconst) >= 10
    ORDER BY median_rating DESC, movies_count DESC
    LIMIT 50
),
directors AS (
    SELECT n.primaryname AS person, 'director' AS role, COUNT(DISTINCT b.tconst) AS movies_count,
           ROUND(AVG(r.averagerating)::numeric, 2) AS avg_rating,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY r.averagerating)::numeric, 2) AS median_rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    JOIN title_principals p ON p.tconst = b.tconst
    JOIN name_basics n ON n.nconst = p.nconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 70000 AND p.category = 'director'
    GROUP BY n.primaryname
    HAVING COUNT(DISTINCT b.tconst) >= 5
    ORDER BY median_rating DESC, movies_count DESC
    LIMIT 50
)
SELECT * FROM actors
UNION ALL
SELECT * FROM directors;
"""
df_top_people = pd.read_sql(query_top_people, engine)
check(df_top_people, "df_top_people")
export(df_top_people, "dl_top_people.csv")

dl_top_people.csv: rows=100, cols=5


## Главный файл для кликов (`dl_movie_cast_region.csv`)

**Это ключевой датасет для интерактивности.**

Он содержит:
- `tconst` и `title` — чтобы связать с таблицей «Топ фильмов».
- `person` и `role` — чтобы показать актёров и режиссёров.
- `region` — чтобы показать регион.

При клике на фильм в DataLens вы свяжете поле `title` из `dl_top_titles` с полем `title` в этом файле — и все данные отфильтруются автоматически.

**Логика:** берём топ-100 фильмов, джойним с `title_principals` (актёры/режиссёры) и `title_akas` (регионы).

In [13]:
query_movie_cast_region = """
WITH top_movies AS (
    SELECT b.tconst, b.primarytitle AS title, b.startyear AS year, r.averagerating AS rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 100000
    ORDER BY r.averagerating DESC, r.numvotes DESC
    LIMIT 100
)
SELECT DISTINCT
    tm.tconst, tm.title, tm.year, tm.rating,
    n.primaryname AS person, p.category AS role, ak.region
FROM top_movies tm
JOIN title_principals p ON p.tconst = tm.tconst
JOIN name_basics n ON n.nconst = p.nconst
LEFT JOIN title_akas ak ON ak.titleid = tm.tconst AND ak.region IS NOT NULL
WHERE p.category IN ('actor', 'actress', 'director')
ORDER BY tm.rating DESC, tm.title, p.category;
"""
df_movie_cast_region = pd.read_sql(query_movie_cast_region, engine)
check(df_movie_cast_region, "df_movie_cast_region")
export(df_movie_cast_region, "dl_movie_cast_region.csv")

dl_movie_cast_region.csv: rows=62903, cols=7


## Связка фильм + жанр (`dl_movie_genre.csv`)

Аналогично прошлому файлу, но для жанров. Позволит при клике на фильм фильтровать график по жанрам.

**Логика:** берём топ-100 фильмов и разворачиваем их жанры в отдельные строки.

In [14]:
query_movie_genre = """
WITH top_movies AS (
    SELECT b.tconst, b.primarytitle AS title, b.startyear AS year, r.averagerating AS rating
    FROM title_basics b
    JOIN title_ratings r ON r.tconst = b.tconst
    WHERE b.titletype = 'movie' AND r.numvotes >= 100000
    ORDER BY r.averagerating DESC, r.numvotes DESC
    LIMIT 100
)
SELECT DISTINCT
    tm.tconst, tm.title, tm.year, tm.rating, g.genre_name AS genre
FROM top_movies tm
JOIN title_genres tg ON tg.tconst = tm.tconst
JOIN genres g ON g.genre_id = tg.genre_id
ORDER BY tm.rating DESC, tm.title;
"""
df_movie_genre = pd.read_sql(query_movie_genre, engine)
check(df_movie_genre, "df_movie_genre")
export(df_movie_genre, "dl_movie_genre.csv")

dl_movie_genre.csv: rows=246, cols=5


## Финальная проверка

Убедимся, что все 10 файлов на месте и имеют корректный размер.

In [15]:
expected = [
    "dl_kpi.csv", "dl_by_year.csv", "dl_by_genre.csv", "dl_by_genre_combo.csv",
    "dl_by_region.csv", "dl_region_genre.csv", "dl_top_titles.csv",
    "dl_top_people.csv", "dl_movie_cast_region.csv", "dl_movie_genre.csv",
]
for f in expected:
    path = PROCESSED_DIR / f
    if path.exists():
        print(f"OK {f} {path.stat().st_size / 1024:.1f} KB")
    else:
        print(f"MISSING {f}")

OK dl_kpi.csv 0.1 KB
OK dl_by_year.csv 3.4 KB
OK dl_by_genre.csv 0.7 KB
OK dl_by_genre_combo.csv 8.5 KB
OK dl_by_region.csv 1.2 KB
OK dl_region_genre.csv 38.2 KB
OK dl_top_titles.csv 6.8 KB
OK dl_top_people.csv 3.4 KB
OK dl_movie_cast_region.csv 3836.8 KB
OK dl_movie_genre.csv 11.0 KB
